## Vectorization

In [1]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer

# Load pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Paths
input_base_dir = 'extracted_texts' 
output_base_dir = 'vectorized_texts' 

def process_and_vectorize_txt_files(input_dir, output_dir):
    """
    This function reads all .txt files from input_dir, vectorizes the content,
    and saves the embeddings in output_dir maintaining the folder structure.
    """
    for foldername, subfolders, filenames in os.walk(input_dir):
        rel_path = os.path.relpath(foldername, input_base_dir)
        output_folder = os.path.join(output_dir, rel_path)
        os.makedirs(output_folder, exist_ok=True)

        for filename in filenames:
            if filename.endswith('.txt'):
                input_file_path = os.path.join(foldername, filename)
                output_file_path = os.path.join(output_folder, filename.replace('.txt', '.npy'))  # Save as .npy

                with open(input_file_path, 'r', encoding='utf-8') as file:
                    text = file.read()

                embedding = model.encode([text])

                np.save(output_file_path, embedding)

                print(f"Processed and saved: {output_file_path}")

# Call the function
process_and_vectorize_txt_files(input_base_dir, output_base_dir)


c:\Users\imed\anaconda3\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
c:\Users\imed\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Processed and saved: vectorized_texts\1995\A1995001.npy
Processed and saved: vectorized_texts\1995\A1995002.npy
Processed and saved: vectorized_texts\1995\A1995003.npy
Processed and saved: vectorized_texts\1995\A1995004.npy
Processed and saved: vectorized_texts\1995\A1995005.npy
Processed and saved: vectorized_texts\1995\A1995006.npy
Processed and saved: vectorized_texts\1995\A1995007.npy
Processed and saved: vectorized_texts\1995\A1995008.npy
Processed and saved: vectorized_texts\1995\A1995009.npy
Processed and saved: vectorized_texts\1995\A1995010.npy
Processed and saved: vectorized_texts\1995\A1995011.npy
Processed and saved: vectorized_texts\1995\A1995012.npy
Processed and saved: vectorized_texts\1995\A1995013.npy
Processed and saved: vectorized_texts\1995\A1995014.npy
Processed and saved: vectorized_texts\1995\A1995015.npy
Processed and saved: vectorized_texts\1995\A1995016.npy
Processed and saved: vectorized_texts\1995\A1995017.npy
Processed and saved: vectorized_texts\1995\A1995

## Save to FAISS vector db

In [2]:
import os
import numpy as np
import faiss

input_base_dir = 'vectorized_texts'
d = 384  # Dimensionality of the embeddings

#Create FAISS index for inner product (for cosine similarity)
index = faiss.IndexFlatIP(d)

def load_and_add_embeddings(input_dir):
    # This function reads .npy vector files from input_dir, normalizes them, and adds them to the FAISS index.
    ids = []
    embeddings = []

    for foldername, _, filenames in os.walk(input_dir):
        for filename in filenames:
            if filename.endswith('.npy'):
                input_file_path = os.path.join(foldername, filename)

                embedding = np.load(input_file_path).flatten()

                embedding = embedding / np.linalg.norm(embedding)

                file_id = os.path.relpath(input_file_path, input_base_dir).replace('\\', '/')
                ids.append(file_id)

                embeddings.append(embedding)

    embeddings_np = np.array(embeddings).astype('float32')

    index.add(embeddings_np)

    return ids

file_ids = load_and_add_embeddings(input_base_dir)

# Save the FAISS index
faiss.write_index(index, 'vector_index_cosine.faiss')
print("FAISS index for cosine similarity saved to 'vector_index_cosine.faiss'")



FAISS index for cosine similarity saved to 'vector_index_cosine.faiss'


## Modeling with Langchain using Gemini-1.5-flash as a start